In [20]:
import h5py
import numpy as np

def extract_mat_file(filepath):
    # Load the file
    f = h5py.File(filepath, 'r')
    
    # Extract BFData
    BFData_dict = {}
    bfdata_group = f['BFData']
    
    # Get the complex arrays (IQ_xAM and IQ_xBmode)
    for array_name in ['IQ_xAM', 'IQ_xBMode']:  # Try different case variations
        if array_name in bfdata_group:
            # Load the data
            data = bfdata_group[array_name][:]
            
            # Check if it's stored as complex or as two separate arrays
            if data.dtype == np.complex64 or data.dtype == np.complex128:
                BFData_dict[array_name] = data
            elif len(data.shape) == 3 and data.shape[-1] == 2:
                # If last dimension is 2 (real, imag), combine to complex
                BFData_dict[array_name] = data[..., 0] + 1j * data[..., 1]
            else:
                BFData_dict[array_name] = data
            print(f"Loaded {array_name} with shape: {BFData_dict[array_name].shape}")
        elif array_name.lower() in [k.lower() for k in bfdata_group.keys()]:
            # Case-insensitive fallback
            actual_name = next(k for k in bfdata_group.keys() if k.lower() == array_name.lower())
            data = bfdata_group[actual_name][:]
            if data.dtype == np.complex64 or data.dtype == np.complex128:
                BFData_dict[actual_name] = data
            elif len(data.shape) == 3 and data.shape[-1] == 2:
                BFData_dict[actual_name] = data[..., 0] + 1j * data[..., 1]
            else:
                BFData_dict[actual_name] = data
            print(f"Loaded {actual_name} with shape: {BFData_dict[actual_name].shape}")
    
    # Extract ReconParams
    ReconParams_dict = {}
    reconparams_group = f['ReconParams']
    
    # Iterate through all items in ReconParams
    for key in reconparams_group.keys():
        item = reconparams_group[key]
        data = item[:] if isinstance(item, h5py.Dataset) else item
        
        # Convert to scalar if it's a 1x1 array
        if isinstance(item, h5py.Dataset) and data.size == 1:
            data = data.item()
        
        ReconParams_dict[key] = data
    
    # Close the h5py file
    f.close()
    
    return BFData_dict, ReconParams_dict

In [21]:
data_dir = "./data/BioUSElective_20260506_JTLRYTAA/ST/"
filepath = data_dir + "data_20260506_164831_3V_ST.mat"

BFData, ReconParams = extract_mat_file(filepath)

Loaded IQ_xAM with shape: (260, 180)
Loaded IQ_xBMode with shape: (260, 180)


In [23]:
ReconParams.keys()

dict_keys(['Angles', 'DemodulationFrequency', 'DemodulationMode', 'FNumber', 'GridNx', 'GridNy', 'GridNz', 'GridOrigin', 'GridScaleX', 'GridScaleY', 'GridScaleZ', 'InterpolationMode', 'NFramesPerBatch', 'NRays', 'NSamplesPerTx', 'NTxPerFrame', 'PiezoXPos', 'PiezoYPos', 'PiezoZPos', 'PointSourceDistOffset', 'PointSourceX', 'PointSourceZ', 'SamplingFrequency', 'SequenceMode', 'SpeedOfSound', 'TXDistOffset', 'TimeCorrectionLens', 'TimeCorrectionStartDepth', 'TimeCorrectionWaveform', 'TransConnector'])